# TradeFlow AI: OLM-Inference Server (T4 x2)

**PENTING:** Pastikan akselerator Kaggle diset ke **GPU T4 x2**.

Model dipecah ke 2 GPU menggunakan tensor parallelism agar fit di memory.

### Changelog
- **v2 (2026-07-15):** Fix CUDA mismatch — pin vLLM + torch cu121, uninstall torchcodec, add env verification

In [ ]:
# ============================================================
# Cell 1: Install Dependencies (Pinned for Kaggle T4 + CUDA 12.x)
# ============================================================
# PENTING: Jangan gunakan `pip install vllm` tanpa version pin!
# Latest vLLM (v0.25.x) menarik PyTorch cu130 yang tidak kompatibel
# dengan Kaggle T4 environment (CUDA 12.x).

# Step 1: Fix Numpy binary incompatibility (PyTorch 2.5+ requires NumPy 2.x)
!pip install -U "numpy>=2.0.0"

# Step 2: Install PyTorch dengan CUDA 12.1 wheels (match Kaggle env)
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121

# Step 3: Install vLLM yang kompatibel dengan T4 + cu121
!pip install "vllm>=0.6.0,<0.7.0" huggingface_hub

# Step 4: Remove torchcodec (dependency baru vLLM yang tidak perlu untuk OCR)
!pip uninstall -y torchcodec

print("\n✅ Dependencies installed successfully")
print("⚠️ PENTING: Lakukan RESTART SESSION / RESTART KERNEL sekarang jika ini baru pertama kali dijalankan!")

In [ ]:
# ============================================================
# Cell 2: Environment Verification (WAJIB sebelum start server)
# ============================================================
import subprocess
import sys

print("=" * 60)
print("🔍 ENVIRONMENT VERIFICATION")
print("=" * 60)

# 1. Check NVIDIA driver & GPU
print("\n--- GPU Info ---")
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

# 2. Check CUDA version
print("\n--- CUDA Version (System) ---")
!nvcc --version 2>/dev/null || echo "nvcc not found (OK jika torch punya bundled CUDA)"

# 3. Check PyTorch CUDA
print("\n--- PyTorch CUDA ---")
import torch
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")
print(f"CUDA version    : {torch.version.cuda}")
print(f"GPU count       : {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

# 4. Check vLLM import
print("\n--- vLLM ---")
try:
    import vllm
    print(f"vLLM version    : {vllm.__version__}")
    print("vLLM import     : ✅ OK")
except Exception as e:
    print(f"vLLM import     : ❌ FAILED — {e}")
    print("\n⚠️  STOP! Perbaiki instalasi sebelum lanjut ke Cell berikutnya.")
    raise SystemExit("vLLM import failed")

# 5. Validation
print("\n" + "=" * 60)
if torch.cuda.is_available() and torch.cuda.device_count() >= 2:
    print("✅ Environment OK — siap menjalankan vLLM server")
elif torch.cuda.is_available() and torch.cuda.device_count() == 1:
    print("⚠️  Hanya 1 GPU terdeteksi. Tensor parallel=2 akan gagal.")
    print("   Pastikan akselerator Kaggle = GPU T4 x2")
else:
    print("❌ Tidak ada GPU! Pastikan notebook menggunakan GPU accelerator.")
print("=" * 60)

In [ ]:
# ============================================================
# Cell 3: Start vLLM Server + Cloudflare Tunnel
# ============================================================
import subprocess
import time
import re
import os
import urllib.request

# 1. Download & install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

# 2. Start vLLM dengan TENSOR PARALLEL = 2 (split model ke 2 GPU)
#    PENTING: dtype=half (float16) karena T4 TIDAK support bfloat16
vllm_cmd = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "allenai/olmOCR-2-7B-1025",
    "--dtype", "half",
    "--tensor-parallel-size", "2",
    "--gpu-memory-utilization", "0.85",
    "--max-model-len", "4096",
    "--port", "8000",
    "--host", "0.0.0.0",
    "--enable-lora",
    "--lora-modules", "cipl_adapter=muhammadghiffari/olm-ocr-cipl-v1",
    "--max-lora-rank", "64"
]

print("\u23f3 Memulai vLLM Server (tensor-parallel=2, 2x T4 GPU)...")
print("   Model akan di-download dan di-split ke 2 GPU (~5-10 menit)...")
vllm_process = subprocess.Popen(
    vllm_cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# 3. Tunggu vLLM benar-benar ready (poll /health endpoint)
print("\u23f3 Menunggu vLLM server ready (polling /health)...")
server_ready = False
poll_deadline = time.time() + 600  # max 10 menit untuk download + load model
while time.time() < poll_deadline:
    # Check apakah process masih hidup
    if vllm_process.poll() is not None:
        print("\n❌ vLLM process terminated unexpectedly!")
        print("\n--- vLLM Logs ---")
        remaining = vllm_process.stdout.read()
        if remaining:
            print(remaining)
        break
    
    try:
        req = urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=3)
        if req.status == 200:
            server_ready = True
            print("\n✅ vLLM Server is READY!")
            break
    except Exception:
        pass
    
    time.sleep(10)
    print(".", end="", flush=True)

if not server_ready:
    print("\n❌ vLLM server gagal start dalam 10 menit.")
    print("   Cek log di bawah untuk detail error.")
    print("\n--- vLLM Server Logs ---")
    try:
        for line in vllm_process.stdout:
            print(line, end="")
    except Exception:
        pass
    raise SystemExit("vLLM failed to start")

# 4. Start Cloudflare Tunnel (hanya setelah server ready)
print("\n\u23f3 Memulai Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:8000"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)

# 5. Baca log tunnel untuk mendapatkan Public URL
url_pattern = re.compile(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com")
public_url = None

deadline = time.time() + 60  # timeout 60 detik
while time.time() < deadline:
    line = tunnel_process.stderr.readline()
    if not line:
        time.sleep(0.5)
        continue
    match = url_pattern.search(line)
    if match:
        public_url = match.group(0)
        print("\n" + "=" * 60)
        print(f"✅ BERHASIL! URL API Anda adalah:")
        print(f"   {public_url}")
        print("")
        print("   Masukkan ke file .env lokal Anda:")
        print(f"   OLM_INFERENCE_URL={public_url}")
        print("=" * 60 + "\n")
        break

if not public_url:
    print("❌ Gagal mendapatkan URL tunnel dalam 60 detik")

# 6. Stream vLLM logs (akan berjalan terus)
print("\n--- vLLM Server Logs (streaming) ---")
try:
    for line in vllm_process.stdout:
        print(line, end="")
except KeyboardInterrupt:
    print("\nStopping...")
    vllm_process.terminate()
    tunnel_process.terminate()